# 01 - Data audit
Structure, types, missingness, duplicates, target balance, outliers and a leakage screen for `application_train.csv`.

> Outputs are cleared in the repository. Run the notebook yourself; with real data present it uses it, otherwise it falls back to clearly labelled synthetic data.

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from credit_risk.config import load_config
from credit_risk.data.loader import load_dataset, dataset_status_report
cfg = load_config()
print(dataset_status_report(cfg))
# demo=True is an EXPLICIT opt-in: if the real Home Credit files are missing, SYNTHETIC data is generated.
df, DATA_SOURCE = load_dataset(cfg, demo=True)
print("\nDATA SOURCE:", DATA_SOURCE, df.shape)
if DATA_SOURCE != "home_credit":
    print("WARNING: SYNTHETIC data - numbers are only a code-path demonstration, not real results.")

In [ ]:
from credit_risk.data.validation import validate_application_data
res = validate_application_data(df)
print('passed:', res.passed); print('errors:', res.errors); print('warnings:', res.warnings); print(res.stats)

In [ ]:
from credit_risk.data.audit import audit_summary, outlier_summary, single_feature_auc
s = audit_summary(df, 'TARGET', 'SK_ID_CURR')
for k in ['shape','n_numeric','n_categorical','duplicate_rows','duplicate_ids','target_distribution','default_rate','imbalance_ratio_neg_to_pos','columns_with_missing','columns_over_50pct_missing','constant_columns','days_employed_sentinel_365243']:
    print(f'{k:32s}', s.get(k))

## Why accuracy is not enough
With a low default rate, predicting 'no default' for everyone gives high accuracy and zero risk discrimination. We therefore track ROC-AUC, PR-AUC, KS, precision/recall at policy thresholds, and Brier/calibration.

In [ ]:
dr = df.TARGET.mean(); print(f'default rate = {dr:.4f}; accuracy of an "approve everyone" rule = {1-dr:.4f}; recall of that rule = 0.0')

In [ ]:
pd.DataFrame(s['categorical_cardinality'].items(), columns=['column','n_unique']).head(10)

## Leakage screen
Univariate AUC per feature (categoricals are target-encoded out-of-fold). Values far from 0.5 (flag at >0.85 or <0.15) are reviewed manually. A flag is a prompt to investigate, not proof of leakage; a low value does not prove absence of leakage.

In [ ]:
leak = single_feature_auc(df, 'TARGET', 'SK_ID_CURR')
print('flagged:', leak.loc[leak.flag!='','feature'].tolist() or 'none')
leak.head(15)

In [ ]:
outlier_summary(df, [c for c in df.select_dtypes('number').columns if c not in ('TARGET','SK_ID_CURR')])